## Gold Layer – ML-Ready Risk Labels

### Step 1: Load Silver Data

In [0]:
silver_df = spark.read.table("workspace.default.silver_heart_disease")
display(silver_df)

### Step 2: Create Gold Layer (ML-Ready Dataset)

Create business/ML labels and final features for risk prediction modeling.

In [0]:
from pyspark.sql.functions import row_number, when, col
from pyspark.sql.window import Window

window_spec = Window.orderBy(col("ingestion_timestamp"))

gold_df = silver_df.withColumn(
    "patient_id", row_number().over(window_spec)
).withColumn(
    "risk_label",
    when(col("heart_disease_flag") == 1, "High Risk").otherwise("Low Risk")
)

display(gold_df)

In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_heart_disease")

### Step 3: Validate Gold Table

In [0]:
%sql
SELECT risk_label, COUNT(*) 
FROM workspace.default.gold_heart_disease 
GROUP BY risk_label;
